# NeurIPS 2026 Smart Buildings — Version 18: Disturbance-Observer Control

**Hypothesis:** Measured zone temperature changes often come from unmodeled solar gains, infiltration, occupancy and mismatched equipment effectiveness. A *disturbance observer (DOB)* can distinguish these changes from the predicted effect of our own previous HVAC commands, and can anticipate near-term comfort violations without a heavy MPC/RLS solve.

## What makes this a DOB

For each valid zone, every 5-minute step uses a deliberately simple nominal discrete thermal model:

```text
T_hat_next = T_prev
           + 0.003*(T_out_prev - T_prev)
           + 0.006*damper_prev*(T_supply_prev - T_prev) [only when AHU on]
           + 0.060*reheat_valve_prev [only when hot water on]
residual = T_measured_now - T_hat_next
d_hat = clip(0.78*d_hat + 0.22*clip(residual,-0.20,+0.20), -0.16,+0.16)
T_compensated = T_measured_now + clip(5*d_hat, -0.80, +0.80)
```

The **nominal input effect is subtracted** before estimating the unknown disturbance. This differentiates V18 from a raw temperature-trend smoother, V4's passive trend forecasting, V6/V12's online RLS/MPC and V2/V3's error-integral feedback. Positive estimated residual triggers earlier cooling; negative residual triggers earlier heating. A temperature gap greater than 2 K in one interval is treated as an unreliable jump, and decays rather than trains the estimate.

## Controller stages

1. **Metadata and observation decoding** — dynamic AHU/zone/actuator indices; zone and outdoor field-specific normalization into Kelvin.
2. **Residual observer** — update one bounded K/step disturbance per zone from *last* temperature, outside temperature and issued damper/reheat/AHU commands; no future information.
3. **Compensation** — shift the effective zone temperature by up to ±0.8 K before setting heating/cooling requests; `DISABLE_OBSERVER=True` is the explicit ablation.
4. **Physical dispatch** — retained V7 AHU air mixing, cooling-setpoint, VAV headroom, fan pressure and hot-water policy so disturbance compensation is the isolated research direction.
5. **State update and safety** — remember the current actuator commands for the **next** sample; clear all observer state on `reset()`; bound observations, updates and actions.

**Scope:** Standalone Kaggle notebook, NumPy and official inference gateway only. Time O(AHUs + zones), memory O(zones), no training, external files or hidden simulator access.

**Limitations:** The nominal gains and 5-step anticipation are heuristic, not system-identified. The disturbance estimate lumps together exogenous heat, actuation errors, stale command-to-plant response and model mismatch; it cannot isolate their physical causes. An observed gain in synthetic tests does **not** demonstrate improved energy/comfort score.

**Evaluation:** Run paired 30-scenario/8,640-step starter-kit evaluation against V7 and the `DISABLE_OBSERVER=True` ablation. Report score, comfort violation, electricity, gas, carbon, cycling, runtime, and observer residual distribution separately; then tune gains only with held-out validation.


In [ ]:
"""Version 18: disturbance-observer-based HVAC control (DOB).

A bounded, discrete-time thermal residual observer estimates lumped unmodeled
zone loads (solar, occupants, infiltration and plant/model mismatch) from the
previous *commanded* HVAC action and the newly measured zone temperature.
The estimated disturbance is converted to a small anticipation correction to
comfort errors before the V7 physical air/water dispatcher runs.

The observer is NOT a moving-average trend estimate: it subtracts nominal
action-conditioned heat transfer before filtering the remaining innovation.
Constants are conservative model assumptions, not identified site parameters.
Runtime: O(AHUs + zones) per five-minute sample, O(zones) state.
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Weather-compensated, thermodynamics-aware air and hot-water dispatch.

    Action contract: continuous float32 in [-1, 1], discrete int32 in {0, 1}.
    All equipment topology and sensor locations come from episode metadata.
    """

    # Physical constants from the starter-kit action descriptions.
    MIN_SUPPLY_K = 285.0
    MAX_SUPPLY_K = 305.0
    MAX_RETURN_FRACTION = 0.70
    ACTIVE_THRESHOLD_K = 0.18
    MIN_VAV_FRACTION = 0.10
    MIN_FAN_PRESSURE_PA = 200.0
    # A cooling-only AHU still admits at least 30% outside air.
    # Increasing intake beyond that floor also raises exhaust-fan power.
    MIN_ECONOMIZER_OUTDOOR_K = 287.0
    MIN_ECONOMIZER_RECIRC = 0.15

    # Five-minute, explicitly action-conditioned nominal zone model.
    # T_next = T + LEAK*(T_out-T) + AIR*damper*(T_supply-T)
    #          + REHEAT*reheat_valve + d
    NOMINAL_LEAK_GAIN = 0.003
    NOMINAL_AIR_GAIN = 0.006
    NOMINAL_REHEAT_GAIN_K = 0.06
    OBSERVER_ALPHA = 0.22
    INNOVATION_LIMIT_K = 0.20
    DISTURBANCE_LIMIT_K_PER_STEP = 0.16
    MAX_SENSOR_STEP_K = 2.0
    ANTICIPATION_STEPS = 5.0
    MAX_CORRECTION_K = 0.80
    # Set True only for an independent V7-like control ablation.
    DISABLE_OBSERVER = False

    def __init__(self, building_meta_data, action_spec):
        """Index observations and actions once, never during each time step."""
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data
        self.n_continuous = int(meta["n_continuous_action"])
        self.n_discrete = int(meta["n_discrete_action"])
        self.fields = {str(s): i for i, s in enumerate(meta["observation_field_names"])}
        self.supply_indices = list(map(int, meta["temp_indices"]))
        self.pressure_indices = list(map(int, meta["pressure_indices"]))
        self.recirc_indices = list(map(int, meta["recirculation_indices"]))
        self.water_indices = list(map(int, meta["water_indices"]))
        self.ahu_count = len(self.supply_indices)
        self.water_supply = self.water_indices[0] if self.water_indices else None
        self.water_pressure = (self.pressure_indices[self.ahu_count]
                               if len(self.pressure_indices) > self.ahu_count else None)
        self.water_run = self.n_discrete - 1

        self.comfort = tuple(map(float, meta["comfort_temp_window"]))
        self.eco = tuple(map(float, meta["eco_temp_window"]))
        self.morning = float(meta["morning_start_hour"])
        self.evening = float(meta["evening_start_hour"])
        normalizer = meta.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {})
        self.zone_mean = float(normalizer.get("sample_mean", 0.0))
        self.zone_std = float(normalizer.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0
        outside_stats = meta.get("observation_normalization_metadata", {}).get(
            "outside_air_temperature_sensor", {})
        self.outdoor_mean = float(outside_stats.get("sample_mean", float("nan")))
        self.outdoor_std = float(outside_stats.get("sample_std", float("nan")))

        def unit_order(key):
            match = re.search(r"\d+", str(key))
            return (int(match.group()) if match else -1, str(key))

        self.ahus = []
        assignments = meta["ahu_zone_assignments"]
        indices = meta["zone_action_indices"]
        for i, identifier in enumerate(sorted(assignments, key=unit_order)[:self.ahu_count]):
            zones = []
            for room in assignments[identifier]:
                vav = str(room).replace("room_", "VAV_")
                temp = self.fields.get(vav + "_zone_air_temperature_sensor")
                if temp is None:
                    continue
                actuators = indices.get(vav, {})
                zones.append({
                    "temperature": temp,
                    "low": self.fields.get(vav + "_zone_air_heating_temperature_setpoint_sensor"),
                    "high": self.fields.get(vav + "_zone_air_cooling_temperature_setpoint_sensor"),
                    "damper": actuators.get("supply_air_damper_percentage_command"),
                    "valve": actuators.get("reheat_valve_percentage_command"),
                })
            self.ahus.append({
                "zones": zones,
                "run": i,
                "supply": self.supply_indices[i],
                "pressure": (self.pressure_indices[i]
                             if i < len(self.pressure_indices) else None),
                "recirc": (self.recirc_indices[i]
                           if i < len(self.recirc_indices) else None),
                "outdoor": self.fields.get(
                    f"ahs_{identifier}_outside_air_temperature_sensor"),
            })
        self.reset()

    def reset(self):
        """Forget prior observations, commands and disturbance estimates."""
        self.disturbance_k_per_step = {}
        self.previous_zone = {}

    @staticmethod
    def _clamp(value, low, high):
        return float(np.clip(value, low, high))

    @classmethod
    def _fraction_action(cls, fraction):
        """Convert physical 0..1 actuator opening to normalized action."""
        return 2.0 * cls._clamp(fraction, 0.0, 1.0) - 1.0

    @staticmethod
    def _temperature_action(kelvin):
        """Supply air 285..305 K maps to -1..1."""
        return float(np.clip((kelvin - 295.0) / 10.0, -1.0, 1.0))

    def _value(self, obs, index):
        """Read an optional sensor without indexing outside the observation."""
        if index is None or not 0 <= index < len(obs):
            return float("nan")
        return float(obs[index])

    def _outside_temperature(self, obs, index):
        """Convert the field-normalized outdoor measurement back to Kelvin."""
        raw = self._value(obs, index)
        if not np.isfinite(raw):
            return float("nan")
        if (np.isfinite(self.outdoor_mean)
                and np.isfinite(self.outdoor_std) and self.outdoor_std > 0.0):
            kelvin = raw * self.outdoor_std + self.outdoor_mean
        else:
            kelvin = raw
        return kelvin if 210.0 <= kelvin <= 330.0 else float("nan")

    def _schedule_fallback(self, obs):
        """Fall back to the occupied/eco schedule if zone setpoints are absent."""
        def clock(sin_name, cos_name, period):
            sine = self._value(obs, self.fields.get(sin_name))
            cosine = self._value(obs, self.fields.get(cos_name))
            if not (np.isfinite(sine) and np.isfinite(cosine)):
                return None
            return float(np.arctan2(sine, cosine) % (2 * np.pi)) * period / (2 * np.pi)

        hour = clock("hod_sin_000", "hod_cos_000", 24.0)
        day = clock("dow_sin_000", "dow_cos_000", 7.0)
        weekday = day is not None and int(day) < 5
        occupied = (weekday and hour is not None
                    and self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zone(self, obs, zone, fallback):
        """Decode one zone (Kelvin); return None for an invalid temperature.

        The temperature observation is standardized. Setpoint observations
        are already in Kelvin and take precedence over schedule fallback.
        """
        raw = self._value(obs, zone["temperature"])
        temperature = raw * self.zone_std + self.zone_mean
        if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
            return None
        low = self._value(obs, zone["low"])
        high = self._value(obs, zone["high"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            low, high = fallback
        return {
            "key": int(zone["temperature"]),
            "actions": zone,
            "temp": temperature,
            "low": float(low),
            "high": float(high),
            "hot": max(0.0, temperature - high),
            "cold": max(0.0, low - temperature),
        }

    def _observe_disturbances(self, statuses):
        """Update the *lumped disturbance* from a one-step model residual.

        Only the last issued commands are used. Invalid/jumped sensors do not
        train the observer; a jump halves any stale disturbance estimate.
        """
        if self.DISABLE_OBSERVER:
            return
        for status in statuses:
            key = status["key"]
            prev = self.previous_zone.get(key)
            if prev is None:
                continue  # No valid actuation/temperature pair yet.
            delta = status["temp"] - prev["temperature"]
            if not np.isfinite(delta) or abs(delta) > self.MAX_SENSOR_STEP_K:
                self.disturbance_k_per_step[key] = (
                    0.5 * self.disturbance_k_per_step.get(key, 0.0))
                continue
            ambient = (prev["outdoor"] if np.isfinite(prev["outdoor"])
                       else prev["temperature"])
            nominal_delta = self.NOMINAL_LEAK_GAIN * (
                ambient - prev["temperature"])
            if prev["supply"] is not None:
                nominal_delta += self.NOMINAL_AIR_GAIN * prev["damper"] * (
                    prev["supply"] - prev["temperature"])
            nominal_delta += self.NOMINAL_REHEAT_GAIN_K * prev["valve"]
            innovation = self._clamp(
                delta - nominal_delta,
                -self.INNOVATION_LIMIT_K, self.INNOVATION_LIMIT_K)
            old = self.disturbance_k_per_step.get(key, 0.0)
            self.disturbance_k_per_step[key] = self._clamp(
                (1.0 - self.OBSERVER_ALPHA) * old
                + self.OBSERVER_ALPHA * innovation,
                -self.DISTURBANCE_LIMIT_K_PER_STEP,
                self.DISTURBANCE_LIMIT_K_PER_STEP)

    def _apply_disturbance_compensation(self, statuses):
        """Convert estimated K/step into a bounded virtual-zone error.

        Positive residual heat increases cooling requests and decreases heat
        requests; negative residual heat does the opposite. This changes the
        *control* before dispatch, not merely a diagnostic or logged value.
        """
        if self.DISABLE_OBSERVER:
            return
        for status in statuses:
            forecast_offset = self._clamp(
                self.ANTICIPATION_STEPS * self.disturbance_k_per_step.get(
                    status["key"], 0.0),
                -self.MAX_CORRECTION_K, self.MAX_CORRECTION_K)
            predicted = status["temp"] + forecast_offset
            status["hot"] = max(0.0, predicted - status["high"])
            status["cold"] = max(0.0, status["low"] - predicted)

    def _remember_issued_actions(self, dispatched, continuous, discrete):
        """Keep one bounded previous-command record per valid zone.

        Inactive AHUs and inactive hot-water plants contribute no HVAC heat
        in the nominal model, regardless of their idle continuous commands.
        """
        previous = {}
        water_on = 0 <= self.water_run < len(discrete) and discrete[self.water_run] == 1
        for statuses, outside, delivered, running in dispatched:
            for status in statuses:
                zone = status["actions"]
                damper_index, valve_index = zone["damper"], zone["valve"]
                damper = (self._clamp((float(continuous[damper_index]) + 1.0) / 2.0,
                                      0.0, 1.0)
                          if running and damper_index is not None
                          and 0 <= damper_index < len(continuous) else 0.0)
                valve = (self._clamp((float(continuous[valve_index]) + 1.0) / 2.0,
                                     0.0, 1.0)
                         if running and water_on and valve_index is not None
                         and 0 <= valve_index < len(continuous) else 0.0)
                previous[status["key"]] = {
                    "temperature": status["temp"],
                    "outdoor": outside,
                    "supply": delivered if running else None,
                    "damper": damper,
                    "valve": valve,
                }
        self.previous_zone = previous
        # Missing/invalid observations cannot carry old estimates indefinitely.
        self.disturbance_k_per_step = {
            key: value for key, value in self.disturbance_k_per_step.items()
            if key in previous
        }

    def _make_idle_actions(self):
        """A zero action is half-open, so explicitly set idle commands."""
        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        for index in self.supply_indices:
            continuous[index] = 1.0  # Warmest: cooling coil effectively idle.
        for index in self.recirc_indices:
            continuous[index] = 1.0  # Max return air, physically capped at 70%.
        return continuous, discrete

    def _select_air_supply(self, zones, outside_k):
        """Select coil setpoint and air mixing from the plant's heat balance.

        Returns (coil_setpoint_K, return_fraction, delivered_K).

        The cooling coil can only LOWER the mixed-air temperature.
        Economizer mode is enabled only if extra outside air can entirely
        avoid mechanical cooling. Otherwise the guaranteed extra exhaust
        airflow can erase a marginal chiller saving; stay at the 70%
        return-air limit until that trade-off has been measured.
        """
        hottest = max(zone["hot"] for zone in zones)
        return_k = float(np.mean([zone["temp"] for zone in zones]))

        if hottest < self.ACTIVE_THRESHOLD_K:
            coil_setpoint_k = self.MAX_SUPPLY_K
        else:
            hottest_zones = [zone["temp"] for zone in zones
                             if zone["hot"] > 0.0]
            mean_hot_k = float(np.mean(hottest_zones))
            coil_setpoint_k = self._clamp(
                mean_hot_k - 4.0 - 1.3 * hottest,
                self.MIN_SUPPLY_K, 297.0)

        if not np.isfinite(outside_k):
            outside_k = return_k

        return_fraction = self.MAX_RETURN_FRACTION
        if (hottest >= self.ACTIVE_THRESHOLD_K
                and self.MIN_ECONOMIZER_OUTDOOR_K <= outside_k < return_k - 1.5):
            temperature_span = return_k - outside_k
            # The lowest mix reachable at 15% return air must be cool
            # enough to meet the coil setpoint without compressor work.
            min_mix_k = (self.MIN_ECONOMIZER_RECIRC * return_k
                         + (1.0 - self.MIN_ECONOMIZER_RECIRC) * outside_k)
            if min_mix_k <= coil_setpoint_k:
                required_return = (coil_setpoint_k - outside_k) / temperature_span
                return_fraction = self._clamp(
                    required_return,
                    self.MIN_ECONOMIZER_RECIRC, self.MAX_RETURN_FRACTION)

        mixed_k = (return_fraction * return_k
                   + (1.0 - return_fraction) * outside_k)
        delivered_k = min(mixed_k, coil_setpoint_k)
        return coil_setpoint_k, return_fraction, delivered_k

    def _control_zone_boxes(self, statuses, delivered, hot_water_demands, continuous):
        """Size VAV flow using actual heating/cooling temperature headroom.

        In contrast to a fixed damper = gain * temperature-error rule,
        requested damper opening falls as cooling headroom grows. This is
        a *proxy* for delivered flow, not a calibrated CFM calculation.
        Returns (mean_damper, peak_damper) for AHU fan coordination.
        """
        sum_damper = 0.0
        peak_damper = 0.0
        for status in statuses:
            zone = status["actions"]
            hot, cold, temperature = (
                status["hot"], status["cold"], status["temp"])
            if hot >= self.ACTIVE_THRESHOLD_K:
                headroom = max(2.5, temperature - delivered)
                damper = self._clamp(0.12 + 2.1 * hot / headroom,
                                     self.MIN_VAV_FRACTION, 0.98)
            elif cold >= self.ACTIVE_THRESHOLD_K:
                # Heating is VAV reheat, never the AHU cooling coil.
                damper = self._clamp(0.22 + 0.21 * cold, 0.18, 0.88)
                if zone["valve"] is not None:
                    valve = self._clamp(0.16 + 0.31 * cold, 0.0, 1.0)
                    continuous[zone["valve"]] = self._fraction_action(valve)
                    hot_water_demands.append((cold, valve))
            else:
                damper = self.MIN_VAV_FRACTION
            sum_damper += damper
            peak_damper = max(peak_damper, damper)
            if zone["damper"] is not None:
                continuous[zone["damper"]] = self._fraction_action(damper)
        # Position fractions are proxies, not calibrated volumetric flow.
        return sum_damper / len(statuses), peak_damper

    def _control_ahu(self, ahu, statuses, outdoor, continuous, discrete, water):
        """Run only if a zone needs conditioning; never call the AHU to heat."""
        if not statuses:
            return
        worst_violation_k = max(max(z["hot"], z["cold"]) for z in statuses)
        if worst_violation_k < self.ACTIVE_THRESHOLD_K:
            return
        target, recirc, delivered = self._select_air_supply(statuses, outdoor)
        mean_damper, peak_damper = self._control_zone_boxes(
            statuses, delivered, water, continuous)
        discrete[ahu["run"]] = 1
        continuous[ahu["supply"]] = self._temperature_action(target)
        if ahu["recirc"] is not None:
            continuous[ahu["recirc"]] = self._fraction_action(recirc)
        if ahu["pressure"] is not None:
            # Use both typical load and worst VAV. A badly overheated room
            # must not disappear in the average of many neutral zones.
            # The fan model uses airflow ~ sqrt(static pressure).
            demand_proxy = max(mean_damper, 0.50 * peak_damper)
            pressure_pa = self.MIN_FAN_PRESSURE_PA + 1250.0 * np.sqrt(
                self._clamp(
                    (demand_proxy - self.MIN_VAV_FRACTION) / 0.65,
                    0.0, 1.0))
            continuous[ahu["pressure"]] = self._clamp(
                pressure_pa / 1000.0 - 1.0, -1.0, 1.0)
        return delivered

    def _control_hot_water(self, water, outdoor_values, continuous, discrete):
        """Weather-compensated hot-water reset, independent of cooling mode.

        Cold outdoor air forces a lower mixed-air temperature because the
        simulator caps recirculation at 70%; compensate with hotter reheat
        water. Only start the pump if a VAV actually requests heating.
        """
        if not water:
            return
        if 0 <= self.water_run < self.n_discrete:
            discrete[self.water_run] = 1
        else:
            return
        worst = max(shortfall for shortfall, _ in water)
        average_valve = sum(valve for _, valve in water) / len(water)
        outside = (sum(outdoor_values) / len(outdoor_values)
                   if outdoor_values else 294.0)
        water_kelvin = self._clamp(
            318.0 + 0.35 * max(0.0, 294.0 - outside) + 5.0 * worst,
            320.0, 349.0)
        if self.water_supply is not None:
            continuous[self.water_supply] = self._fraction_action(
                (water_kelvin - 310.0) / 40.0)
        if self.water_pressure is not None:
            continuous[self.water_pressure] = self._fraction_action(
                self._clamp(0.38 + 0.52 * average_valve, 0.38, 0.95))

    def get_action(self, obs):
        """Observe prior thermal residuals -> compensate -> dispatch -> remember.

        Each residual uses T[t] - (T[t-1] + nominal_effect(action[t-1])).
        There is no search, batch inference, external model, or optimizer.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._make_idle_actions()
        fallback = self._schedule_fallback(obs)
        hot_water_demands, outdoors, dispatched = [], [], []
        for ahu in self.ahus:
            statuses = [self._read_zone(obs, z, fallback)
                        for z in ahu["zones"]]
            statuses = [status for status in statuses if status is not None]
            outdoor = self._outside_temperature(obs, ahu["outdoor"])
            if np.isfinite(outdoor):
                outdoors.append(outdoor)
            self._observe_disturbances(statuses)
            self._apply_disturbance_compensation(statuses)
            delivered = self._control_ahu(
                ahu, statuses, outdoor, continuous, discrete, hot_water_demands)
            running = bool(discrete[ahu["run"]])
            dispatched.append((statuses, outdoor, delivered, running))
        self._control_hot_water(hot_water_demands, outdoors, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        self._remember_issued_actions(dispatched, continuous, discrete)
        return {"continuous_action": continuous, "discrete_action": discrete}


In [ ]:
# Official Kaggle evaluation gateway. Do not replace with a local CSV writer.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
